In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error
import time

### **Loading data**

In [3]:
processed_dir = '../data/processed'
df = pd.read_parquet(f'{processed_dir}/master_data.parquet')

CF only works on "warm" items, so we exclude them (pt 6 of EDA)

In [5]:
item_counts = df.groupby('movieId').size()
warm_movies = item_counts[item_counts > 10].index

cf_df = df[df['movieId'].isin(warm_movies)].copy()

### **Time-based split**

In [7]:
cf_df = cf_df.sort_values('datetime')

split_index = int(len(cf_df) * 0.8)
train_df = cf_df.iloc[:split_index]
test_df = cf_df.iloc[split_index:]

### **Creating centered user-item matrix**

In [8]:
train_matrix = train_df.pivot(index='userId', columns='movieId', values='rating')

User-mean centering to normalize harsh and enthusiastic users (pt 3 of EDA)

In [9]:
user_means = train_matrix.mean(axis=1)
train_matrix_centered = train_matrix.sub(user_means, axis=0)

Fill unrated items with 0 (which now represents the user's average rating)

In [10]:
train_matrix_centered = train_matrix_centered.fillna(0)

### **Train SVD baseline**

In [14]:
start_time = time.time()
# Compress the sparse matrix into 20 latent features
n_latent_features = 20
svd = TruncatedSVD(n_components=n_latent_features, random_state=42)
# Fit and transform to get the user feature matrix
user_factors = svd.fit_transform(train_matrix_centered)
item_factors = svd.components_

stop_time = time.time()
print(f"Training time: {time.time() - start_time:.2f} seconds.\n")

Training time: 0.10 seconds.



### **Reconstructing and evaluating**

Multiply factors back together

In [15]:
reconstructed_matrix = np.dot(user_factors, item_factors)

Convert back to df and add the user means back

In [16]:
pred_df = pd.DataFrame(reconstructed_matrix, index=train_matrix.index, columns=train_matrix.columns)
pred_df = pred_df.add(user_means, axis=0)

Clipping predictions to the 0.5 - 5.0 scale

In [17]:
pred_df = pred_df.clip(lower=0.5, upper=5.0)

Evaluation againt the test set

In [18]:
y_true = []
y_pred = []

for _, row in test_df.iterrows():
    u = row['userId']
    m = row['movieId']
    #Only predict if both user and movie were in training
    if u in pred_df.index and m in pred_df.columns:
        y_true.append(row['rating'])
        y_pred.append(pred_df.loc[u, m])

RMSE

In [19]:
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"Baseline SVD RMSE: {rmse:.4f}")

Baseline SVD RMSE: 0.9805
